# Logits Extractor Module

In [ ]:
def default_params(): 
    return {
        'current_model': 'M1',
        'gpu': True,
        'quantization': 'none', #['none',"int4", "int8", "float32", "float16"]
        'dataset': {
            'path': '/workspaces/CodeSmells/semeru-datasets/code_smells/extraction',
            'transformation': 'curated',
            'content_column': 'code',
            'sampling_size': 500,
        },
        'logging_path': '/workspaces/CodeSmells/datax/code_smells/logs', 
        'callbacks_dir' : '/workspaces/CodeSmells/datax/code_smells/callbacks',
        'cache_dir': '/workspaces/CodeSmells/datax/hugging_face_cache',
        'causal_models': {
            'M1' : 'codellama/CodeLlama-7b-hf', #https://huggingface.co/codellama/CodeLlama-7b-hf, 
            'M2' : 'mistralai/Mistral-7B-v0.3', #https://huggingface.co/mistralai/Mistral-7B-v0.3,
            'M3' : 'microsoft/Phi-3.5-mini-instruct', #https://huggingface.co/microsoft/Phi-3.5-mini-instruct 
            'M4' : 'Qwen/Qwen2.5-Coder-7B', #https://huggingface.co/Qwen/Qwen2.5-Coder-7B
            'M5' : 'facebook/incoder-6B', #https://huggingface.co/facebook/incoder-6B
            'M6' : 'bigcode/starcoder2-7b', #https://huggingface.co/bigcode/starcoder2-7b 
            'M7' : 'deepseek-ai/DeepSeek-R1-Distill-Llama-8B', #https://huggingface.co/deepseek-ai/DeepSeek-R1-Distill-Llama-8B
            'M8' : 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B', #https://huggingface.co/deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B
        },
    }
params = default_params()


#### Imports

In [2]:
import pandas as pd
import os
import time
import numpy as np
import torch
import gc
import seaborn as sns
from scipy import stats
from statistics import NormalDist
import matplotlib.pyplot as plt

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

In [4]:
def create_folder(path):
    if not os.path.exists(path):
        os.makedirs(path)

In [5]:
# Define log file path
log_file = f"{params['logging_path']}/{params['current_model']}/{params['dataset']['transformation']}"
create_folder(log_file)
log_file += '/log.txt'

# Create the log file if it doesn't exist
if not os.path.exists(log_file):
    with open(log_file, 'w'): 
        pass  # Create an empty log file

In [6]:
import logging
logging.basicConfig(filename=log_file, format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)

#### GPU

In [7]:
! nvidia-smi

Tue Feb 18 16:17:00 2025       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 470.103.01   Driver Version: 470.103.01   CUDA Version: 12.3     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  NVIDIA A100-PCI...  Off  | 00000000:61:00.0 Off |                    0 |
| N/A   32C    P0    35W / 250W |      0MiB / 40536MiB |      0%      Default |
|                               |                      |             Disabled |
+-------------------------------+----------------------+----------------------+
                                                                               
+-------

In [8]:
torch.__version__

'2.1.2+cu121'

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() and params['gpu'] else "cpu")
device

device(type='cuda', index=0)

In [10]:
torch.cuda.memory_allocated()

0

## Logits Extractor
>
> Extracting Tensor Logits from a given Neural Code Model
>

#### Model Loading

In [11]:
def instantiate_llm(model_name:str, cache_dir:str):
     '''Instantiate AutoModelForCausalLM'''
     tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir = cache_dir)
     logging.info("Loaded AutoTokenizer - " + model_name)
     model = None
     if params['quantization'] == 'int4':
          model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, load_in_4bit=True)
     elif params['quantization'] == 'int8':
          model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, load_in_8bit=True)
     elif params['quantization'] == 'float32':
          model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, torch_dtype=torch.float32)
     elif params['quantization'] == 'float16':
          model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, torch_dtype=torch.float16)
     else: 
          model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir)
     logging.info("Loaded AutoModelForCausalLM - " + model_name)

     return tokenizer, model

In [12]:
tokenizer, model = instantiate_llm(params['causal_models'][params['current_model']], params['cache_dir'])

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [13]:
model.config

LlamaConfig {
  "_name_or_path": "codellama/CodeLlama-7b-hf",
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 1,
  "eos_token_id": 2,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 11008,
  "max_position_embeddings": 16384,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 32,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_scaling": null,
  "rope_theta": 1000000,
  "tie_word_embeddings": false,
  "torch_dtype": "bfloat16",
  "transformers_version": "4.36.2",
  "use_cache": true,
  "vocab_size": 32016
}

In [14]:
model.to(device) #WARNING, Verify the device before assigning to memory

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32016, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaSdpaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (v_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=11008, bias=False)
          (up_proj): Linear(in_features=4096, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm()
        (post_attention_layernorm): LlamaRMSNorm()
      )
    )
    (norm): LlamaRMSNorm()
  )
  (lm_head):

#### Dataset

In [ ]:
df_dataset = pd.read_json(f"{params['dataset']['path']}/{params['dataset']['transformation']}_{params['dataset']['sampling_size']}.json", )

In [ ]:
#df_dataset = df_dataset[df_dataset['input_lenght']>=700]
#df_dataset = df_dataset[:20]

In [ ]:
df_dataset.head(5)

,id,commit_id,repo,path,file_name,fun_name,commit_message,code,url,language,...,n_ast_nodes,n_identifiers,s_msg_id,s_line,s_column,s_end_line,s_end_column,s_code,category,input_lenght
2268,113061,993109bb3ff1a7acda79227a4d3289d9a4dca81b,nni,test/ut/retiarii/test_oneshot_supermodules.py,test_oneshot_supermodules.py,test_mixed_conv2d,One-shot documentation (more imprv.) (#4924),def test_mixed_conv2d():\n conv = Conv2d(Va...,https://github.com/microsoft/nni.git,Python,...,673,28,C0305,25,0,25,0,,Convention,700
11767,11147,13edc16d806fb5d77a6849551178ccc75937f25f,jina,tests/unit/serve/runtimes/gateway/graph/test_t...,test_topology_graph.py,test_topology_graph_build_two_joins,refactor: rename pod to deployment (#4230)\n\n...,def test_topology_graph_build_two_joins(two_jo...,https://github.com/jina-ai/jina.git,Python,...,515,22,C0305,46,0,46,0,,Convention,700
23353,244286,7b03639942495003eb6e81a2eb16b9915fc56f50,mmdetection,tests/test_utils/test_memory.py,test_memory.py,test_avoidoom,[Feature] Add AvoidOOM to avoid OOM (#7434)\n\...,def test_avoidoom():\n tensor = torch.from_...,https://github.com/open-mmlab/mmdetection.git,Python,...,524,21,C0103,26,8,26,20,AvoidCudaOOM,Convention,700
37906,15590,57a9cb65ab1e947e99f5bc7cc55c4582d232336d,ccxt,python/ccxt/phemex.py,phemex.py,fetch_currencies,1.67.40\n\n[ci skip],"def fetch_currencies(self, params={}):\n ...",https://github.com/ccxt/ccxt.git,Python,...,463,31,C0103,32,16,32,31,precisionString,Convention,700
93630,156019,cccb9d8d8e33a891396b1275c2448c352ef40c27,dask,dask/array/core.py,core.py,__getitem__,absolufy-imports - No relative - PEP8 (#8796)\...,"def __getitem__(self, index):\n # Field...",https://github.com/dask/dask.git,Python,...,674,49,R1705,17,12,24,64,if dt.shape:\n new_axis = list(...,Refactor,700
137873,77684,d21cdad9764056e2e20619a5a847820742d393e3,wagtail,wagtail/admin/views/chooser.py,chooser.py,search,Convert search results view to wagtail.admin.u...,"def search(request, parent_page_id=None):\n ...",https://github.com/wagtail/wagtail.git,Python,...,566,54,W0707,7,8,7,21,raise Http404,Warning,700
152608,21842,cd5a9683be69c86c8f3adcd13385a9bc5db198ec,pipenv,pipenv/core.py,core.py,import_requirements,Rename notpip to pip. Vendor in pip-22.2.1 and...,"def import_requirements(project, r=None, dev=F...",https://github.com/pypa/pipenv.git,Python,...,492,58,R0914,0,0,0,23,def import_requirements,Refactor,700
165236,123398,bacf18832aa4f54c0f6c28dc58e89a89fb1f4338,sqlmap,thirdparty/chardet/sbcharsetprober.py,sbcharsetprober.py,feed,Update of 3rd party library chardet,"def feed(self, byte_str):\n if not self...",https://github.com/sqlmapproject/sqlmap.git,Python,...,420,34,W0511,10,13,10,77,"XXX: This was SYMBOL_CAT_ORDER before, with a...",Warning,700
216363,106440,538ec65ba7634bb9ad9f8eb4ce72713c673969dc,youtube-dl,youtube_dl/jsinterp.py,jsinterp.py,_separate,[jsinterp] Handle regexp literals and throw/ca...,"def _separate(cls, expr, delim=',', max_split=...",https://github.com/ytdl-org/youtube-dl.git,Python,...,625,35,R0912,0,0,0,13,def _separate,Refactor,700
224164,15590,57a9cb65ab1e947e99f5bc7cc55c4582d232336d,ccxt,python/ccxt/phemex.py,phemex.py,fetch_currencies,1.67.40\n\n[ci skip],"def fetch_currencies(self, params={}):\n ...",https://github.com/ccxt/ccxt.git,Python,...,463,31,C0200,19,8,57,13,"for i in range(0, len(currencies)):\n ...",Convention,700


#### Logit Inference

In [25]:
def logit_extractor(model, batch, tf_encoded_inputs, from_index=0):
    """
    Output is the class CausalLMOutputWithPast (https://huggingface.co/transformers/v4.10.1/main_classes/output.html?highlight=causallmoutputwithpast)"
    logits (torch.FloatTensor of shape (batch_size, sequence_length, config.vocab_size)) – Prediction scores of the language modeling head (scores for each vocabulary token before SoftMax).
    The expression i.type(torch.LongTensor).to(device) is for casting labels for the loss
    """
    callbacks_dir = f"{params['callbacks_dir']}/{params['current_model']}_q_{params['quantization']}/{params['dataset']['transformation']}"
    create_folder(callbacks_dir)
    
    for idx, n in enumerate(range(from_index, len(tf_encoded_inputs), batch)):
        torch.cuda.empty_cache()
        output = []
        for encoded_sample in tf_encoded_inputs[n:n+batch]:
            output.append( 
                model(input_ids = encoded_sample, labels = encoded_sample)
            )
        output_logits = [ o['logits'].detach().to('cpu').numpy() for o in output ]  #Logits Extraction
        output_loss = np.array([ o.loss.detach().to('cpu').numpy() for o in output ])  #Language modeling loss (for next-token prediction).

        #Saving Callbacks
        current_batch = idx + (from_index//batch)
        for jdx, o_logits in enumerate(output_logits):
            np.save(f"{callbacks_dir}/logits_tensor[{jdx+n}]_batch[{current_batch}].npy", o_logits)
        np.save(f"{callbacks_dir}/_loss_batch[{current_batch}].npy", output_loss)
        
        print(f"Batch [{current_batch}] Completed")

        #Memory Released
        for out in output:
            del out.logits
            torch.cuda.empty_cache()
            del out.loss
            torch.cuda.empty_cache()
        for out in output_logits:
            del out
            torch.cuda.empty_cache()
        for out in output_loss:
            del out
            torch.cuda.empty_cache()

In [26]:
#Casting Integers to Tensor Integers. Make sure the tesor is created in a device
#We ignored the parameter attention_mask since we are not using masking here [https://huggingface.co/transformers/v4.10.1/glossary.html#attention-mask]
tf_encoded_inputs = [tokenizer(sample, return_tensors='pt')['input_ids'].to(device) for sample in df_dataset[params['dataset']['content_column']].values]

In [27]:
## ACTUAL EXPERIMENT
## TIME AND MEMORY CONSUMING
logit_extractor(
    model = model,
    batch = 1, 
    tf_encoded_inputs = tf_encoded_inputs, 
    from_index=0
)

Batch [0] Completed
Batch [1] Completed
Batch [2] Completed
Batch [3] Completed
Batch [4] Completed
Batch [5] Completed
Batch [6] Completed
Batch [7] Completed
Batch [8] Completed
Batch [9] Completed
Batch [10] Completed
Batch [11] Completed
Batch [12] Completed
Batch [13] Completed
Batch [14] Completed
Batch [15] Completed
Batch [16] Completed
Batch [17] Completed


In [ ]:
torch.cuda.empty_cache()
gc.collect()
del model

: 